# 数据加载测试

本notebook用于测试PDF文件的加载效果，重点关注：

1. **原始文本提取** - PyPDFLoader提取的原始文本
2. **文本清理效果** - `_clean_pdf_text`函数处理后的文本
3. **清理前后对比** - 可视化清理效果
4. **文本质量统计** - 字符数、行数、段落数等指标
5. **多文件对比** - 测试多个PDF文件的加载效果


In [1]:
import sys
from pathlib import Path
from dotenv import load_dotenv
import re
from typing import Dict, List, Tuple

# 添加项目路径
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

# 加载环境变量（从 configs/env 加载）
env_path = project_root / "configs" / "env"
if env_path.exists():
    load_dotenv(env_path)

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from wxchatrag.settings import get_settings
from wxchatrag.wxhub_loader import (
    iter_pdf_paths, 
    load_pdf_documents, 
    build_channel_indexes,
    _clean_pdf_text  # 导入清理函数用于对比
)

print("✓ 导入完成")


d:\worksoft\Anaconda3\envs\llmtorch\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


✓ 导入完成


## 1. 准备测试数据

获取PDF文件列表，选择要测试的PDF文件。


In [2]:
# 获取配置
settings = get_settings()
wxhub_root = settings.wxhub_root
glob_pattern = settings.pdf_glob_pattern or f"*/{settings.pdf_subdir_name}/*.pdf"

print(f"WXhub 根目录: {wxhub_root}")
print(f"PDF 匹配模式: {glob_pattern}")

# 获取所有 PDF 路径
pdf_paths = iter_pdf_paths(wxhub_root, glob_pattern)

print(f"\n✓ 找到 {len(pdf_paths)} 个 PDF 文件")

# 选择要测试的PDF文件（可以修改索引来选择不同的文件）
test_pdf_index = 0  # 默认测试第一个PDF
if pdf_paths:
    test_pdf_path = pdf_paths[test_pdf_index]
    print(f"✓ 选择测试文件: {test_pdf_path.name}")
    print(f"  完整路径: {test_pdf_path}")
else:
    print("⚠ 警告: 未找到 PDF 文件")
    test_pdf_path = None


WXhub 根目录: F:\AIhub\Agent-RAG\WXdown-RAG\data\WXhub
PDF 匹配模式: */pdf/*.pdf

✓ 找到 1018 个 PDF 文件
✓ 选择测试文件: 2026-02-19-人工智能时代真正重要的技能：你的品味.pdf
  完整路径: F:\AIhub\Agent-RAG\WXdown-RAG\data\WXhub\AI寒武纪\pdf\2026-02-19-人工智能时代真正重要的技能：你的品味.pdf


## 2. 加载原始PDF文本

使用PyPDFLoader直接加载PDF，获取原始提取的文本（未清理）。


In [3]:
def load_raw_pdf_text(pdf_path: Path) -> List[Dict[str, any]]:
    """加载PDF的原始文本（未清理）"""
    loader = PyPDFLoader(str(pdf_path))
    raw_pages = []
    for i, doc in enumerate(loader.load()):
        raw_pages.append({
            'page_num': i + 1,
            'raw_text': doc.page_content,
            'metadata': doc.metadata
        })
    return raw_pages

if test_pdf_path:
    raw_pages = load_raw_pdf_text(test_pdf_path)
    raw_text_all = "\n".join([page['raw_text'] for page in raw_pages])
    
    print(f"✓ 原始PDF加载成功")
    print(f"  页数: {len(raw_pages)}")
    print(f"  总字符数: {len(raw_text_all)}")
    print(f"  总行数: {raw_text_all.count(chr(10)) + 1}")
    print(f"\n原始文本预览（前500字符）:")
    print("=" * 80)
    print(raw_text_all[:1000])
    print("=" * 80)
else:
    raw_pages = []
    raw_text_all = ""


✓ 原始PDF加载成功
  页数: 8
  总字符数: 9512
  总行数: 616

原始文本预览（前500字符）:
人工智能时代真正重要的技能：你的品味
 
2026
年
2
月
19
日
 12:05
AI
寒武纪
↑
阅读之前记得关注
+
星标
 ️
，

，每天才能第一时间接收到更新
在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在
在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在
于你选择制作什么
于你选择制作什么
Paul Graham
（
YC
创始人，著名技术作家）
24
年前这篇关于品味的文章，我觉得非常有意
思，分享给大家，现在
AI
发展一日千里，几乎每天都有热点刷屏，但对我们个人真正重要的是
什么？这篇文章或许能给你答案
创造者的品味
创造者的品味
 (Taste for Makers)
2002
年
年
2
月
月
“……
哥白尼对
[
偏心匀速点
]
的美学异议，是他摒弃托勒密体系的一个根本动机
……”
                                
—— 
托马斯
·
库恩，《哥白尼革命》
“
我们所有人都受过凯利
·
约翰逊（
Kelly Johnson
）的训练，并狂热地笃信他的坚持：一架看
起来漂亮的飞机，飞起来也一定漂亮。
”
—— 
本
·
里奇，《臭鼬工厂》
“
美是首要标准：丑陋的数学在世上无容身之地。
”
—— G.H. 
哈代，《一个数学家的辩白》
最近我和一位在麻省理工学院（
MIT
）任教的朋友聊天。他所在的领域目前非常热门，每年都
会被想读研的学生申请信淹没。
“
他们中很多人看起来很聪明，
”
他说，
“
但我看不出他们是否
有任何形式的
品味
品味
。
”
品味。如今你很少听到这个词了。然而，无论我们如何称呼它，我们仍然需要这个词背后的概
念。我朋友的意思是，他想要的学生不仅仅是优秀的技术人员，还要能利用他们的技术知识设
计出美妙东西的人。
数学家将好的工作称为
“
美
”
，不管是现在还是过去，科学家、工程师、音乐家、建筑师、设计
师、作家和画家也都是如此。他们使用同一个词仅仅是巧合吗？还是说他们的意指确实存在某
种重叠？如果有重叠，我们能否利用一个领域关于
“
美
”
的发现来帮助另一个领域？
对于我们这些设计事物的人来说，这

## 3. 加载清理后的PDF文本

使用项目的`load_pdf_documents`函数加载PDF，获取清理后的文本。

In [4]:
if test_pdf_path:
    # 加载清理后的PDF文档
    channel_indexes = build_channel_indexes(wxhub_root)
    cleaned_docs = load_pdf_documents(
        pdf_paths=[test_pdf_path],
        channel_indexes=channel_indexes
    )
    
    cleaned_text_all = "\n\n".join([doc.page_content for doc in cleaned_docs])
    
    print(f"✓ 清理后PDF加载成功")
    print(f"  页数: {len(cleaned_docs)}")
    print(f"  总字符数: {len(cleaned_text_all)}")
    print(f"  段落数: {cleaned_text_all.count(chr(10) + chr(10)) + 1}")
    
    # 显示元数据
    if cleaned_docs:
        meta = cleaned_docs[0].metadata
        print(f"\n文档元数据:")
        print(f"  标题: {meta.get('title', 'N/A')}")
        print(f"  公众号: {meta.get('channel', 'N/A')}")
        print(f"  日期: {meta.get('date', 'N/A')}")
        print(f"  URL: {meta.get('url', 'N/A')}")
    
    print(f"\n清理后文本预览（前500字符）:")
    print("=" * 80)
    print(cleaned_text_all[:1000])
    print("=" * 80)
else:
    cleaned_docs = []
    cleaned_text_all = ""


✓ 清理后PDF加载成功
  页数: 8
  总字符数: 9308
  段落数: 8

文档元数据:
  标题: 人工智能时代真正重要的技能：你的品味
  公众号: AI寒武纪
  日期: 2026-02-19
  URL: https://mp.weixin.qq.com/s/zMxwBfn2Kvn3EgPfRvYOWw

清理后文本预览（前500字符）:
人工智能时代真正重要的技能：你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标  ️， ，每天才能第一时间接收到更新在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在于你选择制作什么于你选择制作什么 Paul Graham （ YC 创始人，著名技术作家） 24 年前这篇关于品味的文章，我觉得非常有意思，分享给大家，现在 AI 发展一日千里，几乎每天都有热点刷屏，但对我们个人真正重要的是什么？这篇文章或许能给你答案创造者的品味创造者的品味 (Taste for Makers) 2002 年年 2 月月 “…… 哥白尼对 [ 偏心匀速点 ] 的美学异议，是他摒弃托勒密体系的一个根本动机 ……”—— 托马斯 · 库恩，《哥白尼革命》 “ 我们所有人都受过凯利 · 约翰逊（ Kelly Johnson ）的训练，并狂热地笃信他的坚持：一架看起来漂亮的飞机，飞起来也一定漂亮。 ” —— 本 · 里奇，《臭鼬工厂》 “ 美是首要标准：丑陋的数学在世上无容身之地。 ” —— G.H. 哈代，《一个数学家的辩白》

最近我和一位在麻省理工学院（ MIT ）任教的朋友聊天。他所在的领域目前非常热门，每年都会被想读研的学生申请信淹没。 “ 他们中很多人看起来很聪明， ” 他说， “ 但我看不出他们是否有任何形式的品味品味。 ” 品味。如今你很少听到这个词了。然而，无论我们如何称呼它，我们仍然需要这个词背后的概念。我朋友的意思是，他想要的学生不仅仅是优秀的技术人员，还要能利用他们的技术知识设计出美妙东西的人。 数学家将好的工作称为 “ 美 ”，不管是现在还是过去，科学家、工程师、音乐家、建筑师、设计师、作家和画家也都是如此。他们使用同一个词仅仅是巧合吗？还是说他们的意指确实存在某种重

## 4. 文本统计对比

对比清理前后的文本统计信息。

In [5]:
def analyze_text_stats(text: str, label: str) -> Dict[str, any]:
    """分析文本统计信息"""
    lines = text.split('\n')
    paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]
    
    # 统计单行字符数
    line_lengths = [len(line) for line in lines if line.strip()]
    avg_line_length = sum(line_lengths) / len(line_lengths) if line_lengths else 0
    
    # 统计段落长度
    para_lengths = [len(p) for p in paragraphs]
    avg_para_length = sum(para_lengths) / len(para_lengths) if para_lengths else 0
    
    # 统计中文字符数
    chinese_chars = len(re.findall(r'[\u4e00-\u9fff]', text))
    
    # 统计连续换行数
    consecutive_newlines = len(re.findall(r'\n{3,}', text))
    
    return {
        'label': label,
        'total_chars': len(text),
        'total_lines': len(lines),
        'non_empty_lines': len([l for l in lines if l.strip()]),
        'paragraphs': len(paragraphs),
        'avg_line_length': round(avg_line_length, 2),
        'avg_para_length': round(avg_para_length, 2),
        'chinese_chars': chinese_chars,
        'consecutive_newlines': consecutive_newlines,
        'spaces': text.count(' '),
        'tabs': text.count('\t'),
    }

if test_pdf_path and raw_text_all and cleaned_text_all:
    raw_stats = analyze_text_stats(raw_text_all, "原始文本")
    cleaned_stats = analyze_text_stats(cleaned_text_all, "清理后文本")
    
    print("文本统计对比:")
    print("=" * 80)
    print(f"{'指标':<20} {'原始文本':<20} {'清理后文本':<20} {'变化':<20}")
    print("-" * 80)
    
    metrics = [
        ('总字符数', 'total_chars'),
        ('总行数', 'total_lines'),
        ('非空行数', 'non_empty_lines'),
        ('段落数', 'paragraphs'),
        ('平均行长度', 'avg_line_length'),
        ('平均段落长度', 'avg_para_length'),
        ('中文字符数', 'chinese_chars'),
        ('连续换行数', 'consecutive_newlines'),
        ('空格数', 'spaces'),
    ]
    
    for metric_name, metric_key in metrics:
        raw_val = raw_stats[metric_key]
        cleaned_val = cleaned_stats[metric_key]
        diff = cleaned_val - raw_val
        diff_pct = (diff / raw_val * 100) if raw_val > 0 else 0
        diff_str = f"{diff:+.0f} ({diff_pct:+.1f}%)"
        print(f"{metric_name:<20} {raw_val:<20} {cleaned_val:<20} {diff_str:<20}")
    
    print("=" * 80)


文本统计对比:
指标                   原始文本                 清理后文本                变化                  
--------------------------------------------------------------------------------
总字符数                 9512                 9308                 -204 (-2.1%)        
总行数                  616                  15                   -601 (-97.6%)       
非空行数                 614                  8                    -606 (-98.7%)       
段落数                  1                    8                    +7 (+700.0%)        
平均行长度                14.44                1161.75              +1147 (+7945.4%)    
平均段落长度               9512.0               1161.75              -8350 (-87.8%)      
中文字符数                7613                 7612                 -1 (-0.0%)          
连续换行数                0                    0                    +0 (+0.0%)          
空格数                  58                   456                  +398 (+686.2%)      


## 5. 清理函数测试

单独测试清理函数，查看清理过程的中间步骤。


In [6]:
if test_pdf_path and raw_pages:
    # 选择一个页面进行详细测试
    test_page_idx = 0
    test_raw_text = raw_pages[test_page_idx]['raw_text']
    
    print(f"测试清理函数（第 {test_page_idx + 1} 页）:")
    print("=" * 80)
    
    # 应用清理函数
    cleaned_result = _clean_pdf_text(test_raw_text)
    
    print(f"\n原始文本长度: {len(test_raw_text)} 字符")
    print(f"清理后长度: {len(cleaned_result)} 字符")
    print(f"变化: {len(cleaned_result) - len(test_raw_text)} "
          f"({(len(cleaned_result) - len(test_raw_text)) / len(test_raw_text) * 100:+.1f}%)")
    
    # 显示原始文本片段
    print(f"\n【原始文本片段】（前300字符）")
    print("-" * 80)
    print(repr(test_raw_text[:300]))
    print("-" * 80)
    
    # 显示清理后文本片段
    print(f"\n【清理后文本片段】（前300字符）")
    print("-" * 80)
    print(repr(cleaned_result[:300]))
    print("-" * 80)
    
    # 显示可读性对比
    print(f"\n【可读性对比】")
    print("-" * 80)
    print("原始文本（前200字符）:")
    print(test_raw_text[:200])
    print("\n清理后文本（前200字符）:")
    print(cleaned_result[:200])
    print("-" * 80)


测试清理函数（第 1 页）:

原始文本长度: 586 字符
清理后长度: 531 字符
变化: -55 (-9.4%)

【原始文本片段】（前300字符）
--------------------------------------------------------------------------------
'人工智能时代真正重要的技能：你的品味\n \n2026\n年\n2\n月\n19\n日\n 12:05\nAI\n寒武纪\n↑\n阅读之前记得关注\n+\n星标\n\x00️\n，\n\x0e\n，每天才能第一时间接收到更新\n在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在\n在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在\n于你选择制作什么\n于你选择制作什么\nPaul Graham\n（\nYC\n创始人，著名技术作家）\n24\n年前这篇关于品味的文章，我觉得非常有意\n思，分享给大家，现在\nAI\n发展一日千里，几乎每天都有热点刷屏，但对我们个人真正重要的是\n什么？这篇文章或许能给你答案\n创造者'
--------------------------------------------------------------------------------

【清理后文本片段】（前300字符）
--------------------------------------------------------------------------------
'人工智能时代真正重要的技能：你的品味2026 年 2 月 19 日 12:05 AI 寒武纪 ↑ 阅读之前记得关注 + 星标 \x00️， \x0e，每天才能第一时间接收到更新在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在在人工智能时代，品味将变得更加重要。当任何人都能制作任何东西时，真正的区别在于你选择制作什么于你选择制作什么 Paul Graham （ YC 创始人，著名技术作家） 24 年前这篇关于品味的文章，我觉得非常有意思，分享给大家，现在 AI 发展一日千里，几乎每天都有热点刷屏，但对我们个人真正重要的是什么？这篇文章或许能给你答案创造者的品味创造者的品味 (Ta'
--------

## 7. 多文件对比

对比多个PDF文件的加载效果，评估清理函数的通用性。


In [7]:
def compare_multiple_pdfs(pdf_paths: List[Path], max_files: int = 5) -> List[Dict]:
    """对比多个PDF文件的加载效果"""
    results = []
    channel_indexes = build_channel_indexes(wxhub_root)
    
    for i, pdf_path in enumerate(pdf_paths[:max_files]):
        print(f"\n处理文件 {i+1}/{min(len(pdf_paths), max_files)}: {pdf_path.name}")
        
        try:
            # 加载原始文本
            loader = PyPDFLoader(str(pdf_path))
            raw_pages = loader.load()
            raw_text = "\n".join([p.page_content for p in raw_pages])
            
            # 加载清理后文本
            cleaned_docs = load_pdf_documents(
                pdf_paths=[pdf_path],
                channel_indexes=channel_indexes
            )
            cleaned_text = "\n\n".join([d.page_content for d in cleaned_docs])
            
            # 提取段落（原始文本按行分割，清理后文本按双换行分割）
            raw_lines = [line.strip() for line in raw_text.split('\n') if line.strip()]
            cleaned_paragraphs = [para.strip() for para in cleaned_text.split('\n\n') if para.strip()]
            
            # 计算统计信息
            raw_stats = analyze_text_stats(raw_text, "原始")
            cleaned_stats = analyze_text_stats(cleaned_text, "清理后")
            
            results.append({
                'file': pdf_path.name,
                'path': str(pdf_path),
                'raw_stats': raw_stats,
                'cleaned_stats': cleaned_stats,
                'pages': len(raw_pages),
                'metadata': cleaned_docs[0].metadata if cleaned_docs else {},
                'raw_text': raw_text,
                'cleaned_text': cleaned_text,
                'raw_lines': raw_lines,
                'cleaned_paragraphs': cleaned_paragraphs
            })
            
            print(f"  ✓ 原始: {raw_stats['total_chars']} 字符, {raw_stats['total_lines']} 行")
            print(f"  ✓ 清理后: {cleaned_stats['total_chars']} 字符, {cleaned_stats['paragraphs']} 段落")
            
        except Exception as e:
            print(f"  ✗ 错误: {e}")
            results.append({
                'file': pdf_path.name,
                'error': str(e)
            })
    
    return results

if pdf_paths:
    print(f"对比前 {min(5, len(pdf_paths))} 个PDF文件:")
    print("=" * 80)
    comparison_results = compare_multiple_pdfs(pdf_paths, max_files=5)
    
    # 汇总统计
    print("\n" + "=" * 80)
    print("汇总统计:")
    print("=" * 80)
    print(f"{'文件名':<40} {'原始字符':<12} {'清理后字符':<12} {'段落数':<10} {'页数':<8}")
    print("-" * 80)
    
    for result in comparison_results:
        if 'error' not in result:
            file_name = result['file'][:38] + "..." if len(result['file']) > 40 else result['file']
            raw_chars = result['raw_stats']['total_chars']
            cleaned_chars = result['cleaned_stats']['total_chars']
            paragraphs = result['cleaned_stats']['paragraphs']
            pages = result['pages']
            print(f"{file_name:<40} {raw_chars:<12} {cleaned_chars:<12} {paragraphs:<10} {pages:<8}")
        else:
            print(f"{result['file']:<40} {'错误':<12}")
    
    print("=" * 80)
    
    # 段落对比
    print("\n" + "=" * 80)
    print("清洗前后段落对比:")
    print("=" * 80)
    
    for idx, result in enumerate(comparison_results, 1):
        if 'error' in result:
            continue
            
        print(f"\n【文件 {idx}: {result['file']}】")
        print("-" * 80)
        
        raw_lines = result.get('raw_lines', [])
        cleaned_paras = result.get('cleaned_paragraphs', [])
        
        # 显示前几个原始行和清理后段落的对比
        max_compare_items = 5  # 最多对比5个段落
        
        print(f"\n原始文本行数: {len(raw_lines)}")
        print(f"清理后段落数: {len(cleaned_paras)}")
        print(f"\n前 {min(max_compare_items, len(raw_lines))} 行原始文本:")
        print("-" * 80)
        for i, line in enumerate(raw_lines[:max_compare_items], 1):
            preview = line[:100] + "..." if len(line) > 100 else line
            print(f"{i}. [{len(line)}字符] {preview}")
        
        print(f"\n前 {min(max_compare_items, len(cleaned_paras))} 个清理后段落:")
        print("-" * 80)
        for i, para in enumerate(cleaned_paras[:max_compare_items], 1):
            preview = para[:200] + "..." if len(para) > 200 else para
            print(f"{i}. [{len(para)}字符] {preview}")
            print()
        
        # 如果段落数较多，显示统计信息
        if len(cleaned_paras) > max_compare_items:
            print(f"... (还有 {len(cleaned_paras) - max_compare_items} 个段落未显示)")
        
        print("=" * 80)
else:
    print("⚠ 没有可用的PDF文件进行对比")


对比前 5 个PDF文件:

处理文件 1/5: 2026-02-19-人工智能时代真正重要的技能：你的品味.pdf
  ✓ 原始: 9512 字符, 616 行
  ✓ 清理后: 9308 字符, 8 段落

处理文件 2/5: 2026-01-28-DeepSeek今年的两个重大更新，一篇详细的总结来了！.pdf
  ✓ 原始: 4641 字符, 626 行
  ✓ 清理后: 4317 字符, 8 段落

处理文件 3/5: 2026-02-08-学AI别再刷朋友圈！AI大神Karpathy的92个信源公布了.pdf
  ✓ 原始: 1938 字符, 208 行
  ✓ 清理后: 1820 字符, 4 段落

处理文件 4/5: 2026-02-08-硅谷顶级风投350页年度报告：从算力竞赛到能源革命，这些科技领域正在剧烈重构.pdf
  ✓ 原始: 11665 字符, 1446 行
  ✓ 清理后: 10482 字符, 15 段落

处理文件 5/5: 2026-01-29-拒绝调包！纯NumPy手搓Ilya推荐的30篇论文，连反向传播都是手写的.pdf
  ✓ 原始: 5372 字符, 543 行
  ✓ 清理后: 4870 字符, 10 段落

汇总统计:
文件名                                      原始字符         清理后字符        段落数        页数      
--------------------------------------------------------------------------------
2026-02-19-人工智能时代真正重要的技能：你的品味.pdf        9512         9308         8          8       
2026-01-28-DeepSeek今年的两个重大更新，一篇详细的总结来了... 4641         4317         8          8       
2026-02-08-学AI别再刷朋友圈！AI大神Karpathy的92个信... 1938         1820         4          4       
2026-02-08-硅谷顶级风投350页年度报告：从